In [4]:
import torch
import os
import numpy as np
import torch.nn as nn

In [3]:
np.array((6,1))

array([6, 1])

<h2>Vectorization and Embeddings</h2>

<p>These vectors are not truly represented as a vector, but as an array of indices (in PyTorch, they are `torch.LongTensor`s), one for each word's index in the vocab.  This representation is not particularly helpful in DNNs since we want continuous representations for each word.</p>

basically creating one hot encoded vector to represent a word or a token to match it up with compact, distributed representation with smaller dimensionallity.

In [42]:
class BERTConfig:
    def __init__(
        self,
        vocab_size = 30522,
        hidden_size = 768,
        num_hiddne_layers = 12,
        num_attention_heads = 12,
        intermediate_size = 3072,
        max_position_embeddings=512,
        dropout=0.1
    ):
        self.vocab_size=vocab_size
        self.hidden_size = 768
        self.num_hidden_layers =12
        self.num_attention_heads = 12
        self.intermediate_size =3072
        self.max_position_embeddings = max_position_embeddings
        self.dropout = dropout

In [43]:
import torch 
import torch.nn as nn

class BERTEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.segment_embeddings = nn.Embedding(2, config.hidden_size)

        self.layer_norm = nn.LayerNorm(config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, attention_mask):
        B, T, C = x.shape 
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        scores = (q@k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        scores += attention_mask
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = (attn@v).transpose(1,2).reshape(B, T, C)
        return self.out(out)

In [44]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.hidden_size % config.num_attention_heads == 0

        self.num_heads = config.num_attention_heads 
        self.head_dim = config.hidden_size 

        self.qkv = nn.Linear(config.hidden_size, config.hidden_size *3)
        self.out = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)


    def forward(self, x, attention_mask):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.premute(2, 0, 3, 1, 4)

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        scores += attention_mask
        attn = torch.softmax(scores, dim =-1)
        attn = self.dropout(attn)

        out = (attn @ v).transpose(1,2).reshape(B, T, C)
        return self.out(out)
        

In [45]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.dropout(self.act(self.fc1(x))))

In [46]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = MultiHeadAttention(config)
        self.ffn = FeedForward(config)

        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.norm2 = nn.LayerNorm(config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, attention_mask):
        x = x + self.dropout(self.attn(self.norm1(x), attention_mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x 

In [47]:
class BERTEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.ModuleList(
        [TransformerBlock(config) for _ in range(config.num_hidden_layers)]
        )

    def forward(self, x, attention_mask):
        for layer in self.layers:
            x = layer(x, attention_mask)
        return x

In [70]:
layers = nn.ModuleList(
        [TransformerBlock(config) for _ in range(config.num_hidden_layers)]
        )
layers[0]

TransformerBlock(
  (attn): MultiHeadAttention(
    (qkv): Linear(in_features=768, out_features=2304, bias=True)
    (out): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (ffn): FeedForward(
    (fc1): Linear(in_features=768, out_features=3072, bias=True)
    (fc2): Linear(in_features=3072, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (act): GELU(approximate='none')
  )
  (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [66]:
class BERT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embeddings = BERTEmbeddings(config)
        self.encoder = BERTEncoder(config)

    def forward(self, input_ids, token_type_ids, attention_mask):
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2) # attention_mask -> 2, 1, 1, 10
        attention_mask = (1.0 - attention_mask) * -1e9

        x = self.embeddings(input_ids, token_type_ids)
        return self.encoder(x, attention_mask)

In [65]:
(1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9

tensor([[[[-0., -0., -0., -0., -0., -0., -0., -0., -0., -0.]]],


        [[[-0., -0., -0., -0., -0., -0., -0., -0., -0., -0.]]]])

In [53]:
config = BERTConfig()
model = BERT(config)

input_ids = torch.randint(0, config.vocab_size, (2,10))
token_type_ids = torch.zeros_like(input_ids)
attention_mask = torch.ones_like(input_ids)


out = model(input_ids, token_type_ids, attention_mask)
print(out.shape)

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
class BERTLearnedPositionalEmbeddings(nn.Module):
    def __init__(self, vocab_dic:int, hidden_dim: int= 768, padding_idx:int = 0, max_seq_len: int = 512):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_dic, hidden_dim, padding_idx)
        self.position_embeddings = nn.Embedding